In [ ]:
# Upload + extract the dataset archive (.rar)
from google.colab import files
up = files.upload()

!apt-get -qq update
!apt-get -qq install -y unrar > /dev/null

import os, glob
rar = list(up.keys())[0]
os.makedirs('data', exist_ok=True)
!unrar x -o+ "$rar" data/ > /dev/null

all_files = sorted([p for p in glob.glob('data/**/*', recursive=True) if os.path.isfile(p)])
print('Extracted files:', len(all_files))
print('Sample files:')
print('\n'.join(all_files[:20]))


In [ ]:
# Read signature points (x,y,t,s) from a text file
import re

def load_points(fp):
    pts = []
    with open(fp, 'r', errors='ignore') as f:
        for ln in f:
            parts = re.split(r'[\s,;]+', ln.strip())
            if len(parts) < 4:
                continue
            try:
                x, y, t, s = map(int, parts[:4])
            except:
                continue
            pts.append((x, y, t, s))
    return pts


In [ ]:
# Split into strokes using pen state (s=1 down, s=0 up)
def strokes_from_points(pts, min_len=3):
    strokes, cur = [], []
    for x, y, t, s in pts:
        if s == 1:
            cur.append((x, y, t, s))
        else:
            if len(cur) >= min_len:
                strokes.append(cur)
            cur = []
    if len(cur) >= min_len:
        strokes.append(cur)
    return strokes


In [ ]:
# Convert stroke to a string-token list using local extrema (xmin/xmax/ymin/ymax)
def extrema_tokens(stroke):
    xs = [p[0] for p in stroke]
    ys = [p[1] for p in stroke]
    out = []
    for i in range(1, len(stroke)-1):
        if xs[i] < xs[i-1] and xs[i] < xs[i+1]: out.append('xmin')
        if xs[i] > xs[i-1] and xs[i] > xs[i+1]: out.append('xmax')
        if ys[i] < ys[i-1] and ys[i] < ys[i+1]: out.append('ymin')
        if ys[i] > ys[i-1] and ys[i] > ys[i+1]: out.append('ymax')
    return out

def signature_sequence(fp):
    pts = load_points(fp)
    seq = []
    for st in strokes_from_points(pts):
        tok = extrema_tokens(st)
        if tok:
            seq.append(tok)
    return seq


In [ ]:
# Levenshtein distance for symbol lists
def lev(a, b):
    m, n = len(a), len(b)
    prev = list(range(n+1))
    cur = [0]*(n+1)
    for i in range(1, m+1):
        cur[0] = i
        ai = a[i-1]
        for j in range(1, n+1):
            cost = 0 if ai == b[j-1] else 1
            cur[j] = min(cur[j-1] + 1, prev[j] + 1, prev[j-1] + cost)
        prev, cur = cur, prev
    return prev[n]


In [ ]:
# Adapted MED between signature stroke-sequences
def med_signature(seqA, seqB):
    m, n = len(seqA), len(seqB)
    D = [[0]*(n+1) for _ in range(m+1)]

    for i in range(1, m+1):
        D[i][0] = D[i-1][0] + len(seqA[i-1])
    for j in range(1, n+1):
        D[0][j] = D[0][j-1] + len(seqB[j-1])

    for i in range(1, m+1):
        a = seqA[i-1]
        la = len(a)
        for j in range(1, n+1):
            b = seqB[j-1]
            lb = len(b)
            D[i][j] = min(
                D[i-1][j] + la,           # delete stroke
                D[i][j-1] + lb,           # insert stroke
                D[i-1][j-1] + lev(a, b)   # replace stroke
            )
    return D[m][n]

def dist_norm(seqA, seqB):
    denom = sum(len(s) for s in seqA) + sum(len(s) for s in seqB)
    return (med_signature(seqA, seqB) / denom) if denom > 0 else 0.0

def score(fp1, fp2):
    return dist_norm(signature_sequence(fp1), signature_sequence(fp2))


In [ ]:
# Auto-detect sets (U1, U2, ...) and sample indices (S1, S2, ...)
import os

def parse_set_sample(path):
    name = os.path.basename(path)
    m = re.search(r'(?i)(u\d+)\s*[^\d]*s\s*(\d+)', name)
    if m:
        return m.group(1).upper(), int(m.group(2))
    # try folder name like U1
    parts = os.path.normpath(path).split(os.sep)
    for part in parts:
        if re.fullmatch(r'(?i)u\d+', part):
            return part.upper(), None
    return 'UNK', None

sig_files = [p for p in all_files if os.path.splitext(p)[1].lower() in {'.txt','.dat','.csv'}]
by_set = {}
for p in sig_files:
    s, idx = parse_set_sample(p)
    by_set.setdefault(s, []).append((idx, p))

sets = sorted([s for s in by_set if s != 'UNK'], key=lambda x: int(x[1:])) + (['UNK'] if 'UNK' in by_set else [])
print('Detected sets:', sets)
for s in sets:
    print(s, 'files:', len(by_set[s]))


In [ ]:
# Evaluate: Genuine vs Genuine and Genuine vs Forged (if available)
THRESHOLD = 0.35

def split_genuine_forged(pairs):
    # pairs: (idx, path); if idx exists and max>=40, use 1..20 genuine, 21..40 forged
    idxs = [i for i,_ in pairs if i is not None]
    if idxs and max(idxs) >= 40:
        genu = [p for i,p in pairs if i is not None and 1 <= i <= 20]
        forg = [p for i,p in pairs if i is not None and 21 <= i <= 40]
        rest = [p for i,p in pairs if i is None or i > 40]
        return genu, forg, rest
    # fallback: all files are 'genuine' (no explicit forgeries)
    return [p for _,p in pairs], [], []

def report_pair(label, a, b, d):
    dec = 'ACCEPT' if d <= THRESHOLD else 'REJECT'
    print(f'{label}: {os.path.basename(a)} vs {os.path.basename(b)} -> {d:.4f}  {dec}')
    return dec

for s in sets:
    pairs = sorted(by_set[s], key=lambda t: (t[0] is None, t[0] if t[0] is not None else 10**9, t[1]))
    genu, forg, extra = split_genuine_forged(pairs)

    print('\n' + '='*55)
    print(f'=== {s} ===')
    print('Total files:', len(pairs), '| genuine:', len(genu), '| forged:', len(forg), '| extra:', len(extra))

    if len(genu) < 2:
        print('(Not enough genuine samples to compare)')
        continue

    # pick template as first genuine
    template = genu[0]
    tseq = signature_sequence(template)
    print('Claimed set:', s)
    print('Template file:', template)
    print('Template tokens (first 20):', (tseq[0][:20] if len(tseq) else []))
    print('Template strokes:', len(tseq))

    # genuine vs genuine (all pairs inside genuine)
    print('\n----- Genuine vs Genuine -----')
    gg_accept = gg_reject = 0
    for i in range(len(genu)):
        for j in range(i+1, len(genu)):
            d = score(genu[i], genu[j])
            dec = report_pair('GG', genu[i], genu[j], d)
            gg_accept += (dec == 'ACCEPT')
            gg_reject += (dec == 'REJECT')
    print('GG summary: ACCEPT=', gg_accept, 'REJECT=', gg_reject)

    # genuine vs forged (if forged exists) else impostor from other sets
    if len(forg) > 0:
        print('\n----- Genuine vs Forged -----')
        gf_accept = gf_reject = 0
        for g in genu:
            for f in forg:
                d = score(g, f)
                dec = report_pair('GF', g, f, d)
                gf_accept += (dec == 'ACCEPT')
                gf_reject += (dec == 'REJECT')
        print('GF summary: ACCEPT=', gf_accept, 'REJECT=', gf_reject)
    else:
        # impostor: use first file from next set (if any)
        other_sets = [x for x in sets if x != s and len(by_set.get(x, [])) > 0]
        if other_sets:
            imp_set = other_sets[0]
            imp_path = sorted(by_set[imp_set])[0][1]
            print(f'\n----- Genuine vs Impostor (from {imp_set}) -----')
            gi_accept = gi_reject = 0
            for g in genu:
                d = score(g, imp_path)
                dec = report_pair('GI', g, imp_path, d)
                gi_accept += (dec == 'ACCEPT')
                gi_reject += (dec == 'REJECT')
            print('GI summary: ACCEPT=', gi_accept, 'REJECT=', gi_reject)
        else:
            print('\n(No forged data and no other set for impostor)')
